# GDP Nowcasting with MIDAS

**Module 03 — Notebook 01**  
**Estimated time:** 15 minutes

## Learning Objectives

1. Build a complete MIDAS nowcast for quarterly GDP using monthly Industrial Production
2. Implement three nowcast vintages (1-month, 2-month, 3-month) for a given quarter
3. Compare MIDAS nowcast accuracy to AR(1) and equal-weight benchmarks
4. Produce a forecast evolution plot showing how the nowcast updates with each monthly release

## Prerequisites

- Module 01: MIDAS fundamentals
- Module 02: Estimation and inference
- Guide 01-02: The nowcasting problem, direct vs. iterated strategies

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

## 1. Load Data

In [ ]:
USE_FRED = False

def load_series_from_csv(series_id):
    import os
    candidates = [
        '../resources',
        '../../module_00_foundations/resources',
        '../../../module_00_foundations/resources',
        '../../../../module_00_foundations/resources',
    ]
    filename_map = {
        'GDPC1':  'gdp_quarterly.csv',
        'INDPRO': 'industrial_production_monthly.csv',
    }
    fname = filename_map[series_id]
    for base in candidates:
        path = os.path.join(base, fname)
        if os.path.exists(path):
            df = pd.read_csv(path, index_col='date', parse_dates=True)
            return df.squeeze()
    raise FileNotFoundError(f"Cannot find {fname}")


if USE_FRED:
    from fredapi import Fred
    fred = Fred()
    gdp_raw = fred.get_series('GDPC1', observation_start='2000-01-01')
    ip_raw  = fred.get_series('INDPRO', observation_start='1999-01-01')
    gdp_growth = np.log(gdp_raw).diff().dropna() * 100
    ip_growth  = np.log(ip_raw).diff().dropna() * 100
else:
    gdp_growth = load_series_from_csv('GDPC1')
    ip_growth  = load_series_from_csv('INDPRO')

print(f"GDP growth:  {len(gdp_growth)} quarters ({gdp_growth.index[0].date()} – {gdp_growth.index[-1].date()})")
print(f"IP growth:   {len(ip_growth)} months ({ip_growth.index[0].date()} – {ip_growth.index[-1].date()})")

## 2. Core Functions

We need a ragged-edge aware MIDAS matrix builder that handles `h_missing` missing lags.

In [ ]:
def build_midas_matrix_ragged(y_low_freq, x_high_freq, K, h_missing=0):
    """
    Build MIDAS matrix with ragged edge support.

    h_missing=0: complete quarter (standard)
    h_missing=1: 2-month vintage (most recent month missing)
    h_missing=2: 1-month vintage (2 most recent months missing)

    The effective lag count is K - h_missing.
    """
    if hasattr(y_low_freq.index, 'to_period'):
        y_q = y_low_freq.copy()
        y_q.index = y_low_freq.index.to_period('Q')
    else:
        y_q = y_low_freq.copy()
        y_q.index = pd.PeriodIndex(y_low_freq.index, freq='Q')

    if hasattr(x_high_freq.index, 'to_period'):
        x_m = x_high_freq.copy()
        x_m.index = x_high_freq.index.to_period('M')
    else:
        x_m = x_high_freq.copy()
        x_m.index = pd.PeriodIndex(x_high_freq.index, freq='M')

    K_avail = K - h_missing  # effective available lags
    rows_Y, rows_X = [], []

    for q in y_q.index:
        # The most recent available month is h_missing months before end of quarter
        last_avail = q.asfreq('M', how='end') - h_missing
        lags = [last_avail - i for i in range(K_avail)]

        if any(lag not in x_m.index for lag in lags):
            continue
        if q not in y_q.index:
            continue

        rows_Y.append(y_q[q])
        rows_X.append([x_m[lag] for lag in lags])

    return np.array(rows_Y), np.array(rows_X)


def beta_weights(K, theta1, theta2):
    if theta1 <= 0 or theta2 <= 0:
        return np.ones(K) / K
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, theta1, theta2)
    s = raw.sum()
    return raw / s if s > 1e-12 else np.ones(K) / K


def estimate_midas(Y, X, starts=None):
    """Profile NLS estimation for Beta MIDAS."""
    K = X.shape[1]
    if starts is None:
        starts = [(1.0, 1.0), (1.0, 5.0), (1.5, 4.0), (2.0, 3.0)]

    def psse(theta):
        t1, t2 = theta
        if t1 <= 0.01 or t2 <= 0.01: return 1e10
        w = beta_weights(K, t1, t2)
        xw = X @ w
        xc, yc = xw - xw.mean(), Y - Y.mean()
        ss = np.dot(xc, xc)
        if ss < 1e-12: return np.sum((Y - Y.mean())**2)
        b = np.dot(yc, xc) / ss
        a = Y.mean() - b * xw.mean()
        return np.sum((Y - a - b * xw)**2)

    best_sse, best_res = np.inf, None
    for t0 in starts:
        res = minimize(psse, t0, method='Nelder-Mead',
                       options={'maxiter': 20000, 'xatol': 1e-8})
        if res.fun < best_sse:
            best_sse, best_res = res.fun, res

    t1, t2 = max(best_res.x[0], 0.01), max(best_res.x[1], 0.01)
    w = beta_weights(K, t1, t2)
    xw = X @ w
    xc, yc = xw - xw.mean(), Y - Y.mean()
    beta_ = np.dot(yc, xc) / np.dot(xc, xc)
    alpha_ = Y.mean() - beta_ * xw.mean()
    return {'theta1': t1, 'theta2': t2, 'alpha': alpha_, 'beta': beta_,
            'sse': best_sse, 'weights': w, 'residuals': Y - alpha_ - beta_ * xw, 'xw': xw}


print("Core functions defined. K=12, h_missing options: 0, 1, 2")

## 3. Build MIDAS Matrices for All Three Vintage Points

We build one matrix for each nowcast vintage:
- `h_missing=0`: complete quarter (3 months available)
- `h_missing=1`: 2-month vintage (2 months available)
- `h_missing=2`: 1-month vintage (1 month available)

In [ ]:
K = 12  # 4 quarterly lags

print("Building MIDAS matrices for 3 vintage points...")
midas_by_vintage = {}

for h in [0, 1, 2]:
    Y_h, X_h = build_midas_matrix_ragged(gdp_growth, ip_growth, K, h_missing=h)
    midas_by_vintage[h] = (Y_h, X_h)
    K_eff = K - h
    vintage_name = {0: '3-month (complete)', 1: '2-month', 2: '1-month'}[h]
    print(f"  h={h} ({vintage_name}): T={len(Y_h)}, K_eff={K_eff}")

# Use the complete-quarter dataset as the primary one
Y, X = midas_by_vintage[0]
T = len(Y)
print(f"\nPrimary dataset: T={T}, K={K}")

## 4. Estimate MIDAS for Each Vintage

Each vintage gets its own estimated weight function. The weight shapes should be similar but not identical.

In [ ]:
print("Estimating Beta MIDAS for each vintage point...")
estimates = {}

for h in [0, 1, 2]:
    Y_h, X_h = midas_by_vintage[h]
    est = estimate_midas(Y_h, X_h)
    estimates[h] = est
    r2 = 1 - est['sse'] / np.sum((Y_h - Y_h.mean())**2)
    vintage_name = {0: '3-month', 1: '2-month', 2: '1-month'}[h]
    print(f"  {vintage_name}: theta=({est['theta1']:.3f}, {est['theta2']:.3f}), "
          f"beta={est['beta']:.4f}, R²={r2:.4f}")

In [ ]:
# --- Visualize weight functions by vintage ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c'}
labels = {0: '3-month (K=12)', 1: '2-month (K=11)', 2: '1-month (K=10)'}

ax = axes[0]
for h in [0, 1, 2]:
    est = estimates[h]
    K_eff = K - h
    j = np.arange(K_eff)
    ax.plot(j, est['weights'], color=colors[h], linewidth=2.5,
            marker='o', markersize=5, label=f"{labels[h]}\n(θ₁={est['theta1']:.2f}, θ₂={est['theta2']:.2f})")

ax.axhline(1/K, color='gray', linestyle='--', linewidth=1.2, label=f'Equal weight (1/{K})')
ax.set_xlabel('Lag j (j=0 = most recent available month)')
ax.set_ylabel('Weight')
ax.set_title('Estimated Weight Functions by Vintage')
ax.legend(fontsize=8)

# Annotate quarter boundaries
for qstart, qlab in [(0, 'Front'), (3, 'Q-1'), (6, 'Q-2'), (9, 'Q-3')]:
    ax.axvline(qstart - 0.3, color='gray', linewidth=0.5, alpha=0.5)

# Right: Quarterly weight aggregates by vintage
ax2 = axes[1]
quarter_labels = ['Current Q', 'Q-1', 'Q-2', 'Q-3']
x_pos = np.arange(4)
width = 0.25

for i, h in enumerate([0, 1, 2]):
    est = estimates[h]
    K_eff = K - h
    w = est['weights']
    # Sum weights within each quarter group
    q_weights = []
    for q_idx in range(4):
        start = q_idx * 3
        end = min((q_idx + 1) * 3, K_eff)
        if start >= K_eff:
            q_weights.append(0.0)
        else:
            q_weights.append(w[start:end].sum())

    ax2.bar(x_pos + i * width, q_weights, width, alpha=0.8,
            color=colors[h], label=labels[h])

ax2.set_xticks(x_pos + width)
ax2.set_xticklabels(quarter_labels)
ax2.axhline(0.25, color='gray', linestyle='--', linewidth=1, label='Equal (0.25 per Q)')
ax2.set_ylabel('Aggregate weight per quarter')
ax2.set_title('Quarterly Weight Allocation by Vintage')
ax2.legend(fontsize=8)

plt.suptitle('MIDAS Weight Functions — Three Nowcast Vintages', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('nowcast_weights_by_vintage.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 5. Expanding-Window Nowcast Evaluation

For each vintage point (h=0, 1, 2), run expanding-window cross-validation and compute RMSE. Also compute the AR(1) benchmark and equal-weight MIDAS.

In [ ]:
MIN_TRAIN = 30

def expanding_window_midas(Y, X, min_train=MIN_TRAIN):
    """
    Expanding-window one-step-ahead MIDAS nowcast errors.
    Returns list of squared prediction errors.
    """
    T = len(Y)
    sq_errors = []
    for end in range(min_train, T):
        Y_tr, X_tr = Y[:end], X[:end]
        est = estimate_midas(Y_tr, X_tr, starts=[(1.0, 5.0), (1.5, 4.0)])
        w = beta_weights(X_tr.shape[1], est['theta1'], est['theta2'])
        xw_test = float(X[end] @ w)
        y_hat = est['alpha'] + est['beta'] * xw_test
        sq_errors.append((Y[end] - y_hat)**2)
    return sq_errors


def expanding_window_ols_agg(Y, X, min_train=MIN_TRAIN):
    """Equal-weight aggregation expanding-window."""
    K = X.shape[1]
    w_eq = np.ones(K) / K
    xw = X @ w_eq
    sq_errors = []
    for end in range(min_train, len(Y)):
        lr = LinearRegression().fit(xw[:end].reshape(-1, 1), Y[:end])
        y_hat = lr.predict([[xw[end]]])[0]
        sq_errors.append((Y[end] - y_hat)**2)
    return sq_errors


def expanding_window_ar1(Y, min_train=MIN_TRAIN):
    """AR(1) benchmark expanding-window."""
    sq_errors = []
    for end in range(min_train, len(Y)):
        Y_tr = Y[:end]
        # OLS: Y[t] = a + rho*Y[t-1]
        Z = np.column_stack([np.ones(end-1), Y_tr[:-1]])
        params = np.linalg.lstsq(Z, Y_tr[1:], rcond=None)[0]
        y_hat = params[0] + params[1] * Y[end-1]
        sq_errors.append((Y[end] - y_hat)**2)
    return sq_errors


print("Computing expanding-window RMSE for all models and vintages...")
print("(This runs profile NLS at each step — takes ~2-3 minutes)")

rmse_results = {}
errors_all = {}

# AR(1) benchmark (same Y regardless of vintage)
Y0, X0 = midas_by_vintage[0]
ar1_errs = expanding_window_ar1(Y0)
rmse_results['AR(1) benchmark'] = np.sqrt(np.mean(ar1_errs))
errors_all['AR(1) benchmark'] = ar1_errs

# Equal-weight MIDAS (3-month vintage)
ols_errs = expanding_window_ols_agg(Y0, X0)
rmse_results['Equal-weight (3-month)'] = np.sqrt(np.mean(ols_errs))
errors_all['Equal-weight (3-month)'] = ols_errs

# Beta MIDAS at each vintage
for h in [0, 1, 2]:
    vintage_name = {0: 'Beta MIDAS (3-month)', 1: 'Beta MIDAS (2-month)', 2: 'Beta MIDAS (1-month)'}[h]
    Y_h, X_h = midas_by_vintage[h]
    print(f"  Running {vintage_name}...")
    errs = expanding_window_midas(Y_h, X_h)
    rmse_results[vintage_name] = np.sqrt(np.mean(errs))
    errors_all[vintage_name] = errs

print("\nExpanding-Window RMSE Results:")
print(f"{'Model':<30} {'RMSE':>8} {'vs AR1':>8}")
print("-" * 50)
ar1_rmse = rmse_results['AR(1) benchmark']
for name, rmse in sorted(rmse_results.items(), key=lambda x: x[1]):
    pct = (rmse - ar1_rmse) / ar1_rmse * 100
    pct_str = f"{pct:+.1f}%"
    print(f"{name:<30} {rmse:>8.4f} {pct_str:>8}")

## 6. Forecast Evolution Plot

For the final few quarters in our sample, plot how the nowcast evolves from the 1-month vintage through to the 3-month vintage.

In [ ]:
# --- Compute nowcast path for last N quarters ---
N_eval_quarters = 8  # Last 8 quarters

# We'll use the full training sample for each of the final N quarters
# to produce the nowcast at each vintage
train_end = T - N_eval_quarters  # Train on all but last N quarters

forecast_evolution = []  # List of dicts with h=0,1,2 nowcasts and actual

for q_idx in range(train_end, T):
    actual = Y0[q_idx]  # Actual GDP growth
    row = {'actual': actual, 'q_idx': q_idx}

    for h in [0, 1, 2]:
        Y_h, X_h = midas_by_vintage[h]
        if q_idx >= len(Y_h):
            row[f'h{h}'] = np.nan
            continue
        # Estimate on all prior quarters
        Y_tr = Y_h[:q_idx]
        X_tr = X_h[:q_idx]
        if len(Y_tr) < MIN_TRAIN:
            row[f'h{h}'] = np.nan
            continue
        est = estimate_midas(Y_tr, X_tr, starts=[(1.0, 5.0), (1.5, 4.0)])
        w = beta_weights(X_tr.shape[1], est['theta1'], est['theta2'])
        xw_test = float(X_h[q_idx] @ w)
        row[f'h{h}'] = est['alpha'] + est['beta'] * xw_test

    forecast_evolution.append(row)

fe_df = pd.DataFrame(forecast_evolution)
print("Forecast evolution computed for last", N_eval_quarters, "quarters:")
print(fe_df[['actual', 'h2', 'h1', 'h0']].round(3).to_string())

In [ ]:
# --- Forecast evolution plots ---
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel 1: RMSE bar chart
ax = axes[0, 0]
model_order = ['AR(1) benchmark', 'Equal-weight (3-month)',
               'Beta MIDAS (1-month)', 'Beta MIDAS (2-month)', 'Beta MIDAS (3-month)']
rmses = [rmse_results.get(m, np.nan) for m in model_order]
bar_colors = ['#d62728', '#aec7e8', '#2ca02c', '#ff7f0e', '#1f77b4']
bars = ax.barh(model_order, rmses, color=bar_colors, alpha=0.85)
for bar, rmse in zip(bars, rmses):
    if not np.isnan(rmse):
        ax.text(rmse + 0.003, bar.get_y() + bar.get_height()/2,
                f'{rmse:.4f}', va='center', fontsize=9)
ax.set_xlabel('OOS RMSE')
ax.set_title('Nowcast RMSE by Model and Vintage')

# Panel 2: Cumulative squared error over holdout
ax2 = axes[0, 1]
for name, errs in errors_all.items():
    line_style = '--' if 'AR' in name else '-'
    ax2.plot(np.cumsum(errs), label=name[:25], linestyle=line_style, linewidth=1.8)
ax2.set_xlabel('Out-of-sample observation')
ax2.set_ylabel('Cumulative squared error')
ax2.set_title('Cumulative Forecast Error Over Time')
ax2.legend(fontsize=7)

# Panel 3-4: Forecast evolution for last quarters
valid_rows = fe_df.dropna()
if len(valid_rows) >= 4:
    show_quarters = min(8, len(valid_rows))
    idx = np.arange(show_quarters)

    ax3 = axes[1, 0]
    actuals = valid_rows['actual'].values[-show_quarters:]
    h0_vals = valid_rows['h0'].values[-show_quarters:]
    h1_vals = valid_rows['h1'].values[-show_quarters:]
    h2_vals = valid_rows['h2'].values[-show_quarters:]

    ax3.scatter(idx, actuals, s=100, marker='D', color='black', zorder=5, label='Actual GDP')
    ax3.plot(idx, h2_vals, color='#2ca02c', linewidth=1.5, marker='s', markersize=5,
             linestyle=':', label='1-month nowcast', alpha=0.8)
    ax3.plot(idx, h1_vals, color='#ff7f0e', linewidth=1.5, marker='^', markersize=5,
             linestyle='--', label='2-month nowcast', alpha=0.8)
    ax3.plot(idx, h0_vals, color='#1f77b4', linewidth=2.0, marker='o', markersize=6,
             linestyle='-', label='3-month nowcast')
    ax3.axhline(0, color='gray', linewidth=0.7, alpha=0.5)
    ax3.set_xlabel(f'Quarter (last {show_quarters} evaluation quarters)')
    ax3.set_ylabel('GDP growth (%)')
    ax3.set_title('Nowcast Evolution — Actual vs. Vintages')
    ax3.legend(fontsize=9)
    ax3.set_xticks(idx)

    # Panel 4: Nowcast revision per quarter
    ax4 = axes[1, 1]
    rev_2to1 = h1_vals - h2_vals  # Revision when adding month 2
    rev_1to0 = h0_vals - h1_vals  # Revision when adding month 3
    width = 0.35
    ax4.bar(idx - width/2, rev_2to1, width, color='#ff7f0e', alpha=0.8, label='Adding month 2 (h=1→h=0)')
    ax4.bar(idx + width/2, rev_1to0, width, color='#1f77b4', alpha=0.8, label='Adding month 3 (h=0 completes)')
    ax4.axhline(0, color='black', linewidth=0.8)
    ax4.set_xlabel('Quarter')
    ax4.set_ylabel('Nowcast revision (%)')
    ax4.set_title('Nowcast Revision at Each Monthly Release')
    ax4.legend(fontsize=9)
    ax4.set_xticks(idx)
else:
    axes[1, 0].text(0.5, 0.5, 'Insufficient data for forecast\nevolution plot',
                   ha='center', va='center', transform=axes[1, 0].transAxes)
    axes[1, 1].text(0.5, 0.5, 'Insufficient data for revision plot',
                   ha='center', va='center', transform=axes[1, 1].transAxes)

plt.suptitle(f'GDP Nowcasting with Beta MIDAS (K={K})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gdp_nowcast_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Summary figure saved.")

## 7. Self-Check

In [ ]:
print("Self-check...")
passed = 0
total = 5

# 1. 3-month vintage has more lags than 1-month
K0 = midas_by_vintage[0][1].shape[1]
K2 = midas_by_vintage[2][1].shape[1]
assert K0 > K2, f"FAIL 1: K at h=0 ({K0}) should be > K at h=2 ({K2})"
passed += 1
print(f"  PASS 1: 3-month vintage uses K={K0} lags > 1-month vintage K={K2}")

# 2. Beta weights sum to 1 for each vintage
for h in [0, 1, 2]:
    w = estimates[h]['weights']
    assert abs(w.sum() - 1.0) < 1e-9, f"FAIL 2: h={h} weights sum to {w.sum()}"
passed += 1
print("  PASS 2: Beta weights sum to 1.0 for all three vintages")

# 3. 3-month nowcast should have lowest or equal RMSE vs 2-month and 1-month
rmse_0 = rmse_results.get('Beta MIDAS (3-month)', 1e10)
rmse_1 = rmse_results.get('Beta MIDAS (2-month)', 1e10)
rmse_2 = rmse_results.get('Beta MIDAS (1-month)', 1e10)
assert rmse_0 <= rmse_2 * 1.05, \
    f"FAIL 3: 3-month RMSE ({rmse_0:.4f}) should be <= 1-month RMSE ({rmse_2:.4f})"
passed += 1
print(f"  PASS 3: 3-month RMSE ({rmse_0:.4f}) <= 1-month RMSE ({rmse_2:.4f}) -- more info helps")

# 4. Ragged edge reduces effective K by h
for h in [0, 1, 2]:
    K_eff = midas_by_vintage[h][1].shape[1]
    assert K_eff == K - h, f"FAIL 4: h={h} effective K={K_eff} should be {K-h}"
passed += 1
print(f"  PASS 4: Effective K = K - h_missing for each vintage")

# 5. AR(1) RMSE computed and positive
ar1_rmse = rmse_results['AR(1) benchmark']
assert ar1_rmse > 0, f"FAIL 5: AR(1) RMSE must be positive, got {ar1_rmse}"
passed += 1
print(f"  PASS 5: AR(1) benchmark RMSE = {ar1_rmse:.4f} (positive)")

print(f"\n{'='*40}")
print(f"Self-check: {passed}/{total} passed")
if passed == total:
    print("All checks passed.")

## Summary

| Concept | Implementation |
|---------|---------------|
| Ragged edge | `build_midas_matrix_ragged(h_missing=h)` drops first h lags |
| Direct MIDAS | One model per vintage; each re-estimates theta |
| RMSE by vintage | More months available → lower RMSE |
| Forecast evolution | Nowcast path from 1-month to 3-month vintage |
| AR(1) benchmark | Key comparison; MIDAS should outperform it |

### Next

**Notebook 02:** Ragged edge simulation — examine the ragged edge effect in controlled experiments with known parameters.